# OpenJev

[openjev/openjev](https://huggingface.co/openjev/openjev), served as [openjev-FP8](https://huggingface.co/openjev/openjev-FP8) by `./serve.sh` (start it first, in a terminal). Each question is one forward pass whose answer is read from the option-letter scores. Edit `CASES` to try your own states and questions.

In [1]:
import json
import subprocess
import time

import httpx

URL = "http://127.0.0.1:3000/v1"  # the vendor helper in front of vLLM; serve.sh picks the checkpoint
client = httpx.Client(timeout=120)


def ask(state, questions):
    start = time.perf_counter()
    r = client.post(f"{URL}/systemone", json={"model": "openjev", "state": state, "questions": questions})
    r.raise_for_status()
    return r.json(), (time.perf_counter() - start) * 1000


def vram():
    out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.used,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout.strip()
    return out


In [2]:
print(json.dumps(client.get(f"{URL}/version").json(), indent=1))
print(vram())

{
 "model_dir": "openjev-FP8",
 "T": 0.85,
 "noul_t": 1.829074,
 "noul_bias": 0.0,
 "flags": {
  "perms": 1,
  "stagger": true,
  "loop_break": false,
  "compact": false,
  "compact_cap": 0,
  "layout": "",
  "pad": 0,
  "targeted": true,
  "instr_style": "pyrepr"
 },
 "shim_file": "shim.py",
 "shim_sha256": "81a22f1b1b8912a465059207ef9f60b7c6c16b4de6372305d867efbe38a1987a"
}
NVIDIA GeForce RTX 5090, 29410 MiB, 32607 MiB


In [3]:
# (state, questions). Types: choice (criteria = {option: description or None}), noul (yes/no), score (ordered levels).
CASES = [
    (
        "Customer message: I was charged twice for my order last week and nobody has replied.",
        {
            "route": {"type": "choice", "instructions": "Which team should handle this?",
                      "criteria": {"billing": "charges, refunds, invoices", "shipping": "delivery and tracking",
                                   "technical": "bugs and login problems"}},
            "angry": {"type": "noul", "instructions": "Is the customer angry?"},
            "urgency": {"type": "score", "instructions": "How urgent is this?",
                        "criteria": ["can wait", "this week", "today", "right now"]},
        },
    ),
    (
        # Grounding judge with a subtle mismatch: the answer gets the year right but the amount wrong.
        "Source: The company reported revenue of $4.2 billion for fiscal 2025, up 8% from the prior year.\n"
        "Answer: Revenue in fiscal 2025 was $4.8 billion, an 8% increase.",
        {
            "grounded": {"type": "noul", "instructions": "Is every claim in the answer supported by the source?"},
            "verdict": {"type": "choice", "instructions": "How does the answer relate to the source?",
                        "criteria": {"faithful": None, "contains a wrong number": None,
                                     "adds unsupported claims": None, "off topic": None}},
        },
    ),
    (
        "Post: Selling my old account with 3 years of history, DM me, payment in gift cards only.",
        {
            "policy": {"type": "choice", "instructions": "Which policy does this post violate, if any?",
                       "criteria": {"none": None, "spam": "unsolicited advertising", "account trading": "buying or selling accounts",
                                    "harassment": None, "scam": "likely fraud"}},
        },
    ),
    (
        {"goal": "Change the shipping address on order #1182",
         "page": {"url": "https://shop.example/orders/1182", "title": "Order #1182"},
         "elements": [{"index": 0, "tag": "a", "text": "Track package"},
                      {"index": 1, "tag": "button", "text": "Cancel order"},
                      {"index": 2, "tag": "button", "text": "Edit delivery details"},
                      {"index": 3, "tag": "a", "text": "Contact support"},
                      {"index": 4, "tag": "button", "text": "Buy it again"}]},
        {
            "element": {"type": "choice", "instructions": "Which element should the agent click next to reach the goal?",
                        "criteria": {"0": "Track package", "1": "Cancel order", "2": "Edit delivery details",
                                     "3": "Contact support", "4": "Buy it again"}},
            "done": {"type": "noul", "instructions": "Is the goal already achieved on this page?"},
        },
    ),
]

In [4]:
ask(*CASES[0])  # warm-up, so the timings below are steady-state

for state, questions in CASES:
    result, ms = ask(state, questions)
    print(f"\n{state if isinstance(state, str) else state['goal']}\n  ({ms:.0f} ms, {result['usage']['input_tokens']} prompt tokens)")
    for key, a in result["answers"].items():
        shown = {k: v for k, v in a.items() if k not in ("type", "legend")}
        print(f"  {key}: {json.dumps(shown)}")


Customer message: I was charged twice for my order last week and nobody has replied.
  (915 ms, 251 prompt tokens)
  route: {"choice": "billing", "probabilities": {"billing": 0.9999, "shipping": 0.0, "technical": 0.0}, "confidence": 0.9999}
  angry: {"noul": 0.6183}
  urgency: {"score": 2.0367, "probabilities": {"0": 0.0025, "1": 0.1219, "2": 0.712, "3": 0.1636}, "confidence": 0.7095}



Source: The company reported revenue of $4.2 billion for fiscal 2025, up 8% from the prior year.
Answer: Revenue in fiscal 2025 was $4.8 billion, an 8% increase.
  (700 ms, 236 prompt tokens)
  grounded: {"noul": 0.0357}
  verdict: {"choice": "contains a wrong number", "probabilities": {"faithful": 0.0037, "contains a wrong number": 0.9933, "adds unsupported claims": 0.0028, "off topic": 0.0003}, "confidence": 0.991}



Post: Selling my old account with 3 years of history, DM me, payment in gift cards only.
  (341 ms, 99 prompt tokens)
  policy: {"choice": "account trading", "probabilities": {"none": 0.0028, "spam": 0.001, "account trading": 0.9895, "harassment": 0.0001, "scam": 0.0067}, "confidence": 0.9868}



Change the shipping address on order #1182
  (1277 ms, 439 prompt tokens)
  element: {"choice": "2", "probabilities": {"0": 0.0001, "1": 0.0, "2": 0.9991, "3": 0.0007, "4": 0.0}, "confidence": 0.9989}
  done: {"noul": 0.0199}


In [5]:
print(vram())  # whole card, including what Windows holds

NVIDIA GeForce RTX 5090, 29431 MiB, 32607 MiB
